# SkillUP Coverage Analysis

**Comprehensive Code Coverage Reporting for the SkillUP Intelligent Reasoning System**

This notebook provides detailed test coverage analysis to ensure code quality and identify untested areas.

---

   
## Overview

**What is Code Coverage?**

Code coverage measures the percentage of your codebase that is executed by automated tests. It helps identify:
* Untested code paths
* Areas needing additional test cases
* Potential bugs hiding in uncovered code

**Coverage Metrics**

| Metric | Description |
|--------|-------------|
| **Line Coverage** | Percentage of code lines executed by tests |
| **Branch Coverage** | Percentage of decision branches (if/else) tested |
| **Function Coverage** | Percentage of functions called by tests |

**SkillUP Test Suite Overview**

| Module | Test Files | Test Count (approx) | Priority |
|--------|------------|---------------------|----------|
| **App** | 1 | ~25 tests | Medium |
| **Knowledge Graph** | 1 | ~20 tests | High |
| **Recommender** | 4 | ~45 tests | Critical |
| **Skill Gap** | 1 | ~12 tests | High |
| **Integration** | 1 | ~9 tests | Medium |
| **Total** | **8 files** | **~111 tests** | - |

**Coverage Goals for SkillUP**

| Module | Target Coverage | Current Focus |
|--------|----------------|--------------|
| **Overall** | ≥ 70% | Baseline quality |
| **Knowledge Graph** | ≥ 75% | Neo4j interactions |
| **Skill Gap** | ≥ 70% | Analysis logic |
| **Recommender** | ≥ 80% | Core business logic |
| **App** | ≥ 60% | Integration layer |

**Why These Targets?**
* **Recommender (80%)**: Core recommendation engine with extensive business logic
* **Knowledge Graph (75%)**: Critical data layer interfacing with Neo4j
* **Skill Gap (70%)**: Complex analysis with various edge cases
* **App (60%)**: Integration layer with external dependencies (Streamlit, OpenAI)
* **Overall (70%)**: Ensures baseline test coverage across the entire codebase

**Note**: Some tests are skipped in CI/CD environments (e.g., Neo4j, OpenAI integration tests) to avoid external dependencies.

---

In [0]:
# Install coverage tools if needed
%pip install pytest pytest-cov pytest-mock neo4j networkx sentence-transformers streamlit openai PyMuPDF databricks-sql-connector python-docx --quiet

import os
import sys
import subprocess
import json
from pathlib import Path
from datetime import datetime
import xml.etree.ElementTree as ET

# ============================================================================
# PATH CONFIGURATION (Standard for all notebooks)
# ============================================================================

# Dynamic REPO_ROOT detection (works for any user)
try:
    # Try spark.conf first (works on Serverless)
    notebook_path = spark.conf.get("spark.databricks.notebook.path")
except:
    # Fallback to dbutils for classic compute
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    except:
        # Last resort - use current working directory
        notebook_path = str(Path.cwd())
        print("⚠️  Could not detect notebook path, using current directory")

# Convert notebook path to workspace path and derive repo root
# notebook_path is like: /Users/{username}/skillup/evaluation/notebooks/NotebookName
if notebook_path.startswith("/"):
    workspace_path = Path("/Workspace") / notebook_path.lstrip("/")
else:
    workspace_path = Path(notebook_path)

# Navigate up from notebooks -> evaluation -> skillup (repo root)
REPO_ROOT = workspace_path.parent.parent.parent if "notebooks" in str(workspace_path) else workspace_path

# Source data directory (version controlled)
DATA_DIR = REPO_ROOT / "data"

# Persistent artifact storage (Volumes - shared, not in git)
EVAL_ARTIFACTS = Path("/Volumes/workspace/default/iss-scratchpad/evaluation")
DATA_ARTIFACTS = Path("/Volumes/workspace/default/iss-scratchpad/data")

# Ensure artifact directories exist
EVAL_ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Add skillup to Python path for imports
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("✅ Dependencies installed")
print(f"📁 REPO_ROOT: {str(REPO_ROOT)}")
print(f"📁 DATA_DIR (source): {DATA_DIR}")
print(f"📦 EVAL_ARTIFACTS: {EVAL_ARTIFACTS}")
print(f"📦 DATA_ARTIFACTS: {DATA_ARTIFACTS}")
print(f"🕐 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "="*70)
print("🚀 Coverage Analysis Ready")
print("="*70)

## Generate Coverage Report

Running the full test suite with coverage instrumentation. This generates three report formats:

1. **Terminal Report** (`--cov-report=term-missing`) - Console output with missing line numbers
2. **HTML Report** (`--cov-report=html`) - Interactive browsable report
3. **XML Report** (`--cov-report=xml`) - Machine-readable format for CI/CD

⏱️ **Expected Runtime:** 30-60 seconds depending on test suite size

In [0]:
print("\n" + "="*70)
print("🧪 RUNNING TESTS WITH COVERAGE INSTRUMENTATION")
print("="*70)
print(f"Started at: {datetime.now().strftime('%H:%M:%S')}\n")

start_time = datetime.now()

# Prevent __pycache__ creation in Databricks workspace
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'

# Run pytest with coverage for each module separately
result = subprocess.run([
    "python", "-m", "pytest",
    "tests/",
    "--cov=app",
    "--cov=knowledgegraph",
    "--cov=recommender",
    "--cov=skillgap",
    "--cov-report=term-missing",
    "--cov-report=html",
    "--cov-report=xml",
    "-v",
    "--tb=no",  # Don't show tracebacks to reduce output
    "--continue-on-collection-errors"
], capture_output=True, text=True, cwd=REPO_ROOT, env=env)

end_time = datetime.now()
elapsed = (end_time - start_time).total_seconds()

# Display output (last 100 lines to avoid excessive output)
output_lines = result.stdout.split('\n')
if len(output_lines) > 150:
    print("... (output truncated, showing last 150 lines) ...\n")
    print('\n'.join(output_lines[-150:]))
else:
    print(result.stdout)

if result.stderr:
    stderr_lines = result.stderr.split('\n')[:20]  # Only show first 20 warning lines
    print("\n⚠️ Warnings (first 20):")
    print('\n'.join(stderr_lines))

print("\n" + "="*70)
if result.returncode == 0:
    print(f"✅ Tests completed with coverage in {elapsed:.2f}s")
else:
    print(f"⚠️ Tests completed with some failures in {elapsed:.2f}s")
    print("Note: Coverage data is still collected from passing tests")
print("="*70)

print("\n📊 Reports generated:")
print(f"   • Terminal: See output above")
print(f"   • HTML: {REPO_ROOT}/htmlcov/index.html")
print(f"   • XML: {REPO_ROOT}/coverage.xml")

## Coverage Summary

Parsing coverage data from `coverage.xml` to extract overall and per-module coverage percentages.

In [0]:
from collections import defaultdict

print("\n" + "="*70)
print("📊 PARSING COVERAGE DATA")
print("="*70)

coverage_file = f"{REPO_ROOT}/coverage.xml"
module_coverage = {}
overall_line_rate = 0.0

try:
    tree = ET.parse(coverage_file)
    root = tree.getroot()
    
    # Extract overall coverage
    overall_line_rate = float(root.attrib.get('line-rate', 0)) * 100
    
    # Group files by module
    module_stats = defaultdict(lambda: {'covered': 0, 'total': 0, 'files': []})
    
    # Parse each file (class in XML)
    for package in root.findall('.//package'):
        for cls in package.findall('.//class'):
            filename = cls.attrib.get('filename', '')
            name = cls.attrib.get('name', '')
            line_rate = float(cls.attrib.get('line-rate', 0))
            
            # Count lines from line elements (more accurate)
            lines = cls.findall('.//line')
            total_lines = len(lines)
            covered_lines = sum(1 for line in lines if int(line.attrib.get('hits', 0)) > 0)
            
            if total_lines == 0:  # Skip files with no code
                continue
            
            # Determine module from filename
            # Files are like: knowledgegraph.py, recommender.py, skillgap.py, app.py, config.py etc.
            module = None
            if 'knowledgegraph' in name.lower():
                module = 'Knowledge Graph'
            elif 'recommender' in name.lower():
                module = 'Recommender'
            elif 'skillgap' in name.lower():
                module = 'Skill Gap'
            elif name.lower() in ['app.py', 'config.py', 'conversation_service.py', 'cv_service.py', 
                                   'data_access.py', 'llm_service.py', 'ui_components.py', 
                                   'state.py', 'metrics.py', 'logging_config.py', 'cv_parser.py']:
                module = 'App'
            # Skip test files and other utilities
            elif 'test' in name.lower() or name in ['__init__.py', 'demo_integration.py']:
                continue
            
            if module:
                module_stats[module]['covered'] += covered_lines
                module_stats[module]['total'] += total_lines
                module_stats[module]['files'].append((name, line_rate * 100, covered_lines, total_lines))
    
    # Convert to module_coverage format
    for module, stats in module_stats.items():
        if stats['total'] > 0:
            coverage_pct = (stats['covered'] / stats['total']) * 100
            module_coverage[module] = {
                'coverage': coverage_pct,
                'lines_covered': stats['covered'],
                'lines_total': stats['total'],
                'file_count': len(stats['files'])
            }
    
    print("✅ Coverage data parsed successfully")
    print(f"📈 Overall Coverage: {overall_line_rate:.1f}%")
    print(f"📦 Modules found: {len(module_coverage)}")
    
    if len(module_coverage) > 0:
        print("\n📋 Module breakdown:")
        for module, data in sorted(module_coverage.items()):
            print(f"   • {module}: {data['file_count']} files, {data['lines_total']} lines, {data['coverage']:.1f}% covered")
    else:
        print("\n⚠️ Warning: No module-specific coverage found")
    
except FileNotFoundError:
    print("❌ coverage.xml not found. Run the coverage generation cell first.")
except Exception as e:
    print(f"❌ Error parsing coverage data: {e}")
    import traceback
    traceback.print_exc()

## Coverage by Module

Detailed breakdown showing coverage for each module with lines covered vs. total lines.

In [0]:
print("\n" + "="*70)
print("📦 MODULE COVERAGE BREAKDOWN")
print("="*70)

if not module_coverage:
    print("\n⚠️ No module-specific coverage data available")
    print("\nShowing overall coverage only:")
    print(f"  Overall: {overall_line_rate:.1f}%")
    print("\n💡 To get per-module breakdown, the tests were run with --cov=<module> flags.")
else:
    print(f"\n{'Module':<25} {'Coverage':<12} {'Lines Covered/Total':<25} {'Package'}")
    print("-"*90)
    
    for module_name in ['Knowledge Graph', 'Skill Gap', 'Recommender', 'App']:
        if module_name in module_coverage:
            data = module_coverage[module_name]
            cov_pct = data['coverage']
            lines_cov = data['lines_covered']
            lines_tot = data['lines_total']
            pkg_name = data.get('package_name', 'N/A')
            
            # Color-code based on whether it meets threshold
            targets = {
                'Knowledge Graph': 80.0,
                'Skill Gap': 75.0,
                'Recommender': 85.0,
                'App': 70.0
            }
            target = targets.get(module_name, 0)
            status = "✅" if cov_pct >= target else "❌"
            
            print(f"{module_name:<25} {status} {cov_pct:>5.1f}%      {lines_cov:>5} / {lines_tot:<7}      {pkg_name}")
        else:
            print(f"{module_name:<25} {'N/A':<12} {'N/A':<25} {'Not found'}")
    
    print("="*90)
    print(f"\nOverall Coverage: {overall_line_rate:.1f}%")

## Threshold Validation

Comparing actual coverage against target thresholds to identify modules that need more testing.

In [0]:
print("\n" + "="*70)
print("🎯 THRESHOLD VALIDATION")
print("="*70)

# Define targets (aligned with Overview section)
targets = {
    'Knowledge Graph': 75.0,
    'Skill Gap': 70.0,
    'Recommender': 80.0,
    'App': 60.0
}

print(f"\n{'Module':<25} {'Target':<10} {'Actual':<10} {'Status':<12} {'Gap'}")
print("-"*75)

modules_below_threshold = []
modules_meeting_threshold = []

for module_name in ['Knowledge Graph', 'Skill Gap', 'Recommender', 'App']:
    target = targets[module_name]
    
    if module_name in module_coverage:
        actual = module_coverage[module_name]['coverage']
        
        # Determine status
        if actual >= target:
            status = "✅ PASS"
            gap = f"+{actual - target:.1f}%"
            modules_meeting_threshold.append(module_name)
        else:
            status = "❌ FAIL"
            gap = f"-{target - actual:.1f}%"
            modules_below_threshold.append((module_name, target - actual))
    else:
        actual = 0.0
        status = "❌ NO DATA"
        gap = f"-{target:.1f}%"
        modules_below_threshold.append((module_name, target))
    
    print(f"{module_name:<25} {target:>5.0f}%     {actual:>5.1f}%    {status:<12} {gap}")

print("-"*75)
print(f"{'OVERALL':<25} {'70.0':>5}%     {overall_line_rate:>5.1f}%    ", end="")
if overall_line_rate >= 70.0:
    print(f"✅ PASS       +{overall_line_rate - 70.0:.1f}%")
else:
    print(f"❌ FAIL       -{70.0 - overall_line_rate:.1f}%")
print("="*75)

if modules_below_threshold:
    modules_below_threshold.sort(key=lambda x: x[1], reverse=True)  # Sort by gap size
    print(f"\n⚠️ {len(modules_below_threshold)} modules below threshold\n")
    print("📝 Priority Action Items:")
    for i, (module, missing) in enumerate(modules_below_threshold, 1):
        print(f"   {i}. {module}: Need +{missing:.1f}% coverage")
else:
    print("\n🎉 All modules meet or exceed coverage targets!")

if modules_meeting_threshold:
    print(f"\n✅ {len(modules_meeting_threshold)} modules meeting targets: {', '.join(modules_meeting_threshold)}")

print("\n💡 Tips for Improving Coverage:")
print("   • Focus on edge cases and error handling")
print("   • Add tests for untested functions/methods")
print("   • Review uncovered lines section below")
print("   • Increase branch coverage with conditional tests")

## Uncovered Lines

Identifying critical uncovered code lines from the terminal report to prioritize test writing.

In [0]:
print("\n" + "="*70)
print("🔍 UNCOVERED LINES ANALYSIS")
print("="*70)

# Parse the terminal output to find uncovered lines
if result.returncode is not None:
    lines = result.stdout.split('\n')
    
    # Look for lines with missing coverage (format: "module.py    123    5    96%   45-47, 52")
    uncovered_sections = []
    in_coverage_section = False
    
    for line in lines:
        if 'TOTAL' in line:
            in_coverage_section = False
        if '------' in line:
            in_coverage_section = True
            continue
        
        if in_coverage_section and line.strip():
            # Look for lines with missing coverage indicators
            parts = line.split()
            if len(parts) >= 5 and parts[-1] and any(char.isdigit() for char in parts[-1]):
                if '-' in parts[-1] or ',' in parts[-1]:
                    module_name = parts[0]
                    missing_lines = parts[-1]
                    uncovered_sections.append((module_name, missing_lines))
    
    if uncovered_sections:
        print("\n📋 Files with uncovered lines:\n")
        for module, missing in uncovered_sections[:10]:  # Show top 10
            print(f"   • {module:<40} Missing: {missing}")
        
        if len(uncovered_sections) > 10:
            print(f"\n   ... and {len(uncovered_sections) - 10} more files")
    else:
        print("\n✅ No uncovered lines found, or 100% coverage achieved!")
    
    print("\n💡 Focus on:")
    print("   1. Error handling paths")
    print("   2. Edge cases and boundary conditions")
    print("   3. Exception handling blocks")
    print("   4. Rarely-used code paths")
else:
    print("\n⚠️ Coverage data not available. Run the test cell first.")

## View HTML Report

The HTML report provides an interactive, color-coded view of coverage with drill-down capabilities.

In [0]:
html_report_path = f"{REPO_ROOT}/htmlcov/index.html"

print("\n" + "="*70)
print("🌐 HTML COVERAGE REPORT")
print("="*70)

if os.path.exists(html_report_path):
    print("\n✅ HTML report generated successfully")
    print(f"\n📂 Report Location:")
    print(f"   {html_report_path}")
    
    print("\n📖 How to View:")
    print("   1. Download the htmlcov/ folder from your workspace")
    print("   2. Open index.html in a web browser")
    print("   3. Click through modules and files for detailed line-by-line coverage")
    
    print("\n🎨 Color Legend:")
    print("   🟢 Green = Covered by tests")
    print("   🔴 Red = Not covered (needs tests)")
    print("   🟡 Yellow = Partial coverage (some branches)")
    
    # Count HTML files
    try:
        htmlcov_dir = f"{REPO_ROOT}/htmlcov"
        html_files = [f for f in os.listdir(htmlcov_dir) if f.endswith('.html')]
        print(f"\n📊 Report Statistics:")
        print(f"   • {len(html_files)} HTML files generated")
        print(f"   • Interactive source code viewer")
        print(f"   • Searchable coverage data")
    except Exception as e:
        pass
else:
    print("\n❌ HTML report not found")
    print("   Run the 'Generate Coverage Report' cell first")

print("\n" + "="*70)

## Coverage Trends

Tracking coverage improvements over time (requires historical data).

In [0]:
print("\n" + "="*70)
print("📈 COVERAGE TRENDS OVER TIME")
print("="*70)

# Check for historical coverage data
results_dir = Path(REPO_ROOT) / "evaluation" / "results"
historical_data = []

if results_dir.exists():
    # Look for baseline JSON files with coverage data
    for file in sorted(results_dir.glob("baseline_*.json")):
        try:
            with open(file, 'r') as f:
                data = json.load(f)
                if 'coverage' in data or 'snapshot_date' in data:
                    historical_data.append({
                        'date': data.get('snapshot_date', file.stem),
                        'file': file.name
                    })
        except:
            pass

if historical_data:
    print("\n📊 Historical coverage data found:\n")
    for entry in historical_data[-5:]:  # Show last 5
        print(f"   • {entry['date']}: {entry['file']}")
    
    print("\n💡 To track trends:")
    print("   1. Run this notebook regularly (e.g., weekly)")
    print("   2. Save results with timestamp to evaluation/results/")
    print("   3. Parse historical baseline_*.json files")
    print("   4. Plot coverage % over time")
else:
    print("\n📝 No historical coverage data found yet")
    print("\n🔄 To start tracking trends:")
    print("   1. Run this notebook and save results")
    print("   2. Store coverage data with timestamp:")
    print("      • File: evaluation/results/coverage_YYYYMMDD.json")
    print("      • Format: {\"date\": \"2026-04-18\", \"overall\": 82.5, \"modules\": {...}}")
    print("   3. Re-run this notebook periodically")
    print("   4. Compare results over time")

print("\n📊 Current Baseline:")
print(f"   Date: {datetime.now().strftime('%Y-%m-%d')}")
print(f"   Overall Coverage: {overall_line_rate:.1f}%")

if module_coverage:
    print("   Module Coverage:")
    for module, data in module_coverage.items():
        print(f"      • {module}: {data['coverage']:.1f}%")

print("\n💾 Save this baseline for future comparison!")
print("="*70)